
1. Import config
2. Check if watermark table exists
   → If NO  → first run → read batch_1, set watermark to 2024-01-01
   → If YES → read last_extracted_at from watermark table
              → filter current batch WHERE modified_date > last_extracted_at

3. Write filtered rows to Bronze Delta (APPEND mode)

4. Update watermark table with:
   → last_extracted_at = max(modified_date) from this batch
   → batch_id = current batch
   → rows_extracted = count of rows written
   → status = SUCCESS

In [0]:
%python

%run ../config/pipeline_config

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime

# ── Which batch to read (change per run) ─────────────────────
# STUDY NOTE:
#   In production this comes from the orchestrator parameter.
#   For now we set it manually per run.
#   Run 1 → batch_1, Run 2 → batch_2, Run 3 → batch_3
CURRENT_BATCH = "batch_3"
batch_path    = f"{BRONZE_PATH}/{CURRENT_BATCH}"



In [0]:
%python

# ── Check watermark → decide full load or incremental ────────
# STUDY NOTE:
#   First run detection pattern:
#   Check if watermark table exists AND has a row for this source.
#   If not → full load (no filter on modified_date)
#   If yes → incremental (filter WHERE modified_date > last_extracted_at)
#
#   This pattern ensures the pipeline is self-healing:
#   Delete the watermark row → next run does a full reload

watermark_exists = (
    spark.catalog.tableExists(WATERMARK_TABLE) and
    spark.table(WATERMARK_TABLE)
        .filter(F.col("table_name") == PIPELINE["source_table"])
        .count() > 0
)

if watermark_exists:
    # Get last extracted timestamp for this source table
    last_extracted = spark.table(WATERMARK_TABLE) \
        .filter(F.col("table_name") == PIPELINE["source_table"]) \
        .agg(F.max("last_extracted_at")) \
        .collect()[0][0]

    print(f"Watermark found: {last_extracted}")
    print(f"Reading incremental: modified_date > {last_extracted}")

    df_batch = spark.read.format("delta").load(batch_path) \
        .filter(F.col("modified_date") > last_extracted)
else:
    print("No watermark found — performing full load")
    last_extracted = None

    df_batch = spark.read.format("delta").load(batch_path)

rows_extracted = df_batch.count()
print(f"Rows to extract: {rows_extracted:,}")


In [0]:
%python

# ── Write to Bronze Delta ─────────────────────────────────────
# STUDY NOTE:
#   Bronze always APPENDS — never overwrites.
#   Each extraction adds to the history of what arrived.
#   This means Bronze accumulates across all runs.
#   The modified_date column tells you when each row arrived
#   from the source system.
#
#   Why append and not overwrite?
#   PRO: Full audit trail of every extraction
#   PRO: Can replay Silver from Bronze if SCD logic changes
#   CON: Bronze grows over time → needs VACUUM periodically

BRONZE_TABLE = "default.bronze_customers"

df_batch.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(BRONZE_TABLE)

print(f"✓ Written to Bronze: {BRONZE_TABLE}")
print(f"  Rows appended: {rows_extracted:,}")

In [0]:
%python

# ── Update watermark table ────────────────────────────────────
# STUDY NOTE:
#   After every successful extraction we update the watermark.
#   new_watermark = max(modified_date) from THIS batch.
#   Next run will filter WHERE modified_date > this value.
#
#   We use MERGE not INSERT because:
#   Run 1 → no row exists → INSERT
#   Run 2 → row exists → UPDATE
#   MERGE handles both cases in one operation.
#
#   Edge case: If 0 rows extracted, skip watermark update.
#   This happens when incremental load finds no new data.

if rows_extracted > 0:
    new_watermark = df_batch \
        .agg(F.max("modified_date")) \
        .collect()[0][0]

    print(f"New watermark: {new_watermark}")

    # Create watermark row to upsert
    df_watermark_update = spark.createDataFrame([{
        "table_name":         PIPELINE["source_table"],
        "last_extracted_at":  new_watermark,
        "batch_id":           CURRENT_BATCH,
        "rows_extracted":     rows_extracted,
        "status":             "SUCCESS",
        "updated_at":         datetime.now()
    }])

    # MERGE into watermark table
    if spark.catalog.tableExists(WATERMARK_TABLE):
        DeltaTable.forName(spark, WATERMARK_TABLE) \
            .alias("target") \
            .merge(
                df_watermark_update.alias("source"),
                "target.table_name = source.table_name"
            ) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
    else:
        df_watermark_update.write \
            .format("delta") \
            .saveAsTable(WATERMARK_TABLE)

    print(f"✓ Watermark updated: {PIPELINE['source_table']} → {new_watermark}")
else:
    new_watermark = last_extracted
    print(f"No rows extracted — watermark unchanged: {new_watermark}")

In [0]:
%python

# ── Verify Bronze ─────────────────────────────────────────────
df_bronze = spark.table(BRONZE_TABLE)

print("=" * 50)
print("Bronze Extraction Summary")
print("=" * 50)
print(f"Total Bronze rows : {df_bronze.count():,}")
print(f"Batch extracted   : {CURRENT_BATCH}")
print(f"Rows this run     : {rows_extracted:,}")
print(f"New watermark     : {new_watermark}")
print("=" * 50)

# Show watermark table
print("\nWatermark table:")
spark.table(WATERMARK_TABLE).show(truncate=False)